# Notebook 16 – Cross Validation


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data.csv", encoding="latin1")
df = df.dropna(subset=["Description"]).copy()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["IsCancelled"] = df["InvoiceNo"].astype(str).str.startswith("C").astype(int)
df["AbsQuantity"] = df["Quantity"].abs()
df["Month"] = df["InvoiceDate"].dt.month
df["IsInternational"] = (df["Country"] != "United Kingdom").astype(int)

feature_cols = ["AbsQuantity", "UnitPrice", "Month", "IsInternational"]
X = df[feature_cols]
y = df["IsCancelled"]
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancelled,AbsQuantity,Month,IsInternational
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,0,6,12,0
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,6,12,0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,0,8,12,0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,6,12,0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,0,6,12,0


## Why Cross Validation?

Every earlier notebook evaluated a model using a *single* train/test
split. That's simple, but it has a real weakness: the reported score
depends partly on which specific rows happened to land in the test set.
A "lucky" or "unlucky" split can make a model look better or worse than
it really is.

**Demonstration:** the exact same model is trained and scored on 10
different random train/test splits of the same data. If a single split
were fully trustworthy, these scores would all be nearly identical.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

single_split_scores = []
for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    single_split_scores.append(accuracy_score(y_test, model.predict(X_test)))

print("Accuracy across 10 different random single splits:")
print(np.round(single_split_scores, 4))
print(f"\nRange: {min(single_split_scores):.4f} to {max(single_split_scores):.4f}")
print(f"Standard deviation across splits: {np.std(single_split_scores):.5f}")
print("\nEven this small amount of spread means a single split's score is a bit of a")
print("coin flip -- which specific rows end up in 'test' genuinely changes the number reported.")

Accuracy across 10 different random single splits:
[0.9828 0.9828 0.9829 0.9828 0.9829 0.9829 0.9828 0.9828 0.9828 0.9828]

Range: 0.9828 to 0.9829
Standard deviation across splits: 0.00002

Even this small amount of spread means a single split's score is a bit of a
coin flip -- which specific rows end up in 'test' genuinely changes the number reported.


**Cross Validation** fixes this by evaluating a model on *multiple*
different splits of the data and averaging the results, giving a far
more reliable, less luck-dependent estimate of how the model will
actually perform on new data — and, as a bonus, a sense of how much that
estimate could vary (see "Model Stability" below).

## K-Fold Cross Validation

**What it does:** splits the entire dataset into `K` equally-sized
"folds." The model is trained `K` times, and each time, a *different*
fold is held out as the validation set while the model trains on the
remaining `K - 1` folds. Every row gets used for validation exactly once,
and for training `K - 1` times, across the whole process — so, unlike a
single split, every single row eventually contributes to the reported
score.

In [3]:
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression(max_iter=1000)
kfold_scores = cross_val_score(model, X, y, cv=kfold, scoring="accuracy")

print("Accuracy for each of the 5 folds:")
print(np.round(kfold_scores, 4))
print(f"\nMean accuracy: {kfold_scores.mean():.4f}")
print(f"Standard deviation across folds: {kfold_scores.std():.5f}")

Accuracy for each of the 5 folds:
[0.9833 0.9828 0.9827 0.9827 0.9828]

Mean accuracy: 0.9829
Standard deviation across folds: 0.00023


## Stratified K-Fold

**What it does:** the same idea as K-Fold, with one important addition —
each fold is constructed to preserve the *same class proportions* as the
overall dataset. Plain K-Fold shuffles rows randomly into folds without
regard to class balance, which is a real problem on an imbalanced target
like `IsCancelled` (only ~1.7% positive): a fold could easily end up with
far too few (or even zero) cancelled orders purely by chance, making that
fold's score unreliable or undefined.

**Demonstration:** the class balance within each individual fold is
compared between plain K-Fold and Stratified K-Fold.

In [4]:
from sklearn.model_selection import StratifiedKFold

print("Overall IsCancelled rate in the full dataset:", round(y.mean(), 4))

print("\nPositive-class rate in each PLAIN K-Fold fold:")
for i, (_, val_idx) in enumerate(kfold.split(X, y)):
    print(f"  Fold {i+1}: {y.iloc[val_idx].mean():.4f}")

skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("\nPositive-class rate in each STRATIFIED K-Fold fold:")
for i, (_, val_idx) in enumerate(skfold.split(X, y)):
    print(f"  Fold {i+1}: {y.iloc[val_idx].mean():.4f}")

Overall IsCancelled rate in the full dataset: 0.0172

Positive-class rate in each PLAIN K-Fold fold:
  Fold 1: 0.0168
  Fold 2: 0.0172
  Fold 3: 0.0174
  Fold 4: 0.0173
  Fold 5: 0.0172

Positive-class rate in each STRATIFIED K-Fold fold:
  Fold 1: 0.0172
  Fold 2: 0.0172
  Fold 3: 0.0172
  Fold 4: 0.0172
  Fold 5: 0.0172


In [5]:
skfold_scores = cross_val_score(model, X, y, cv=skfold, scoring="accuracy")
print("Stratified K-Fold accuracy per fold:", np.round(skfold_scores, 4))
print(f"Mean: {skfold_scores.mean():.4f}   Std: {skfold_scores.std():.5f}")
print("\n(On this dataset the plain-vs-stratified difference is small, since even a")
print("random fold of ~100k rows still contains plenty of the minority class -- but")
print("on a smaller dataset, or a more extreme imbalance, this difference can be decisive.)")

Stratified K-Fold accuracy per fold: [0.9829 0.9828 0.9828 0.9829 0.9829]
Mean: 0.9829   Std: 0.00005

(On this dataset the plain-vs-stratified difference is small, since even a
random fold of ~100k rows still contains plenty of the minority class -- but
on a smaller dataset, or a more extreme imbalance, this difference can be decisive.)


## Leave-One-Out (LOO)

**What it does:** the most extreme version of K-Fold — set `K` equal to
the total number of rows, `n`. Each fold consists of exactly *one* row
held out for validation, with the model trained on all `n - 1` remaining
rows. This uses the absolute maximum amount of training data on every
single fit, but requires fitting the model `n` times, which becomes
extremely expensive for anything beyond a small dataset.

**Demonstration:** because fitting a model 100,000+ times is impractical,
this is run on a small random subsample (300 rows) purely to illustrate
the mechanics.

In [6]:
from sklearn.model_selection import LeaveOneOut
import time

small_sample = df.sample(n=300, random_state=42)
X_small = small_sample[feature_cols]
y_small = small_sample["IsCancelled"]

loo = LeaveOneOut()
start = time.time()
loo_scores = cross_val_score(LogisticRegression(max_iter=1000), X_small, y_small, cv=loo, scoring="accuracy")
elapsed = time.time() - start

print(f"Leave-One-Out on {len(X_small)} rows required {loo.get_n_splits(X_small)} separate model fits.")
print(f"Took {elapsed:.2f} seconds for just {len(X_small)} rows.")
print(f"Mean LOO accuracy: {loo_scores.mean():.4f}")
print("\nAt this rate, running LOO on the FULL dataset (~540,000 rows) would require")
print("540,000 separate model fits -- illustrating exactly why LOO is rarely used on")
print("anything beyond small datasets in practice.")

Leave-One-Out on 300 rows required 300 separate model fits.
Took 7.16 seconds for just 300 rows.
Mean LOO accuracy: 0.9900

At this rate, running LOO on the FULL dataset (~540,000 rows) would require
540,000 separate model fits -- illustrating exactly why LOO is rarely used on
anything beyond small datasets in practice.


## Cross-Validation Score

**What it is:** the general term for the result `cross_val_score` (or any
cross-validation procedure) produces — an array of one score per fold,
which is then typically summarized by its **mean** (the headline
performance estimate) and its **standard deviation** (how much that
estimate varies across folds — see "Model Stability" below). Reporting
just the mean, without the spread, throws away useful information about
how confident that estimate actually is.

In [7]:
print(f"{'Strategy':25s} {'Mean Accuracy':>15s} {'Std Dev':>10s}")
print(f"{'Single split (seed=42)':25s} {single_split_scores[0]:15.4f} {'n/a':>10s}")
print(f"{'K-Fold (K=5)':25s} {kfold_scores.mean():15.4f} {kfold_scores.std():10.5f}")
print(f"{'Stratified K-Fold (K=5)':25s} {skfold_scores.mean():15.4f} {skfold_scores.std():10.5f}")

Strategy                    Mean Accuracy    Std Dev
Single split (seed=42)             0.9828        n/a
K-Fold (K=5)                       0.9829    0.00023
Stratified K-Fold (K=5)            0.9829    0.00005


## Training vs. Validation Performance

For every fold in K-Fold or Stratified K-Fold, there are actually *two*
scores available: the model's score on the fold it was trained on
(training performance) and its score on the fold it was held out from
(validation performance). Comparing these two — averaged across all
folds — gives the same overfitting/underfitting diagnostic seen in
Notebook 15, but now computed more reliably across several folds instead
of just one split.

In [8]:
from sklearn.model_selection import cross_validate

cv_results = cross_validate(model, X, y, cv=skfold, scoring="accuracy", return_train_score=True)

print("Per-fold TRAINING accuracy:  ", np.round(cv_results["train_score"], 4))
print("Per-fold VALIDATION accuracy:", np.round(cv_results["test_score"], 4))
print(f"\nMean training accuracy:   {cv_results['train_score'].mean():.4f}")
print(f"Mean validation accuracy: {cv_results['test_score'].mean():.4f}")
print(f"Gap: {cv_results['train_score'].mean() - cv_results['test_score'].mean():.4f}")
print("\nA small gap here suggests Logistic Regression is neither overfitting nor")
print("underfitting badly on this feature set -- consistent with it being a fairly simple,")
print("low-variance model (see Notebook 15's Bias-Variance discussion).")

Per-fold TRAINING accuracy:   [0.9829 0.9829 0.9829 0.9829 0.9829]
Per-fold VALIDATION accuracy: [0.9829 0.9828 0.9828 0.9829 0.9829]

Mean training accuracy:   0.9829
Mean validation accuracy: 0.9829
Gap: 0.0000

A small gap here suggests Logistic Regression is neither overfitting nor
underfitting badly on this feature set -- consistent with it being a fairly simple,
low-variance model (see Notebook 15's Bias-Variance discussion).


## Model Stability

**What it is:** how consistent a model's cross-validation scores are
across different folds — measured directly by the standard deviation of
the per-fold scores. A **stable** model produces nearly the same score
regardless of which fold was held out; an **unstable** model's score
swings noticeably depending on which specific rows it happened to be
evaluated on, which is itself a warning sign (often linked to high
variance, as discussed in Notebook 15).

**Demonstration:** comparing the fold-to-fold stability of a simple,
low-complexity model (a shallow Decision Tree) against a highly complex,
unrestricted one, using the same Stratified K-Fold splits.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

shallow_tree = DecisionTreeClassifier(max_depth=3, random_state=42)
deep_tree = DecisionTreeClassifier(random_state=42)  # unrestricted depth

shallow_scores = cross_val_score(shallow_tree, X, y, cv=skfold, scoring="accuracy")
deep_scores = cross_val_score(deep_tree, X, y, cv=skfold, scoring="accuracy")

print(f"{'Model':30s} {'Fold Scores'}")
print(f"{'Shallow Tree (max_depth=3)':30s} {np.round(shallow_scores, 4)}")
print(f"{'Unrestricted Tree':30s} {np.round(deep_scores, 4)}")

print(f"\n{'Model':30s} {'Mean':>8s} {'Std Dev':>10s}")
print(f"{'Shallow Tree (max_depth=3)':30s} {shallow_scores.mean():8.4f} {shallow_scores.std():10.5f}")
print(f"{'Unrestricted Tree':30s} {deep_scores.mean():8.4f} {deep_scores.std():10.5f}")

## Comparing Model Performance Using Different Validation Strategies

Bringing it all together: the same Logistic Regression model, scored
using every validation strategy covered in this notebook, side by side.

In [ ]:
strategy_results = pd.DataFrame({
    "Strategy": [
        "Single train/test split",
        "K-Fold (K=5)",
        "Stratified K-Fold (K=5)",
        "Leave-One-Out (n=300 subsample)",
    ],
    "Mean Accuracy": [
        np.mean(single_split_scores),
        kfold_scores.mean(),
        skfold_scores.mean(),
        loo_scores.mean(),
    ],
    "Std Dev Across Folds/Splits": [
        np.std(single_split_scores),
        kfold_scores.std(),
        skfold_scores.std(),
        loo_scores.std(),
    ],
    "Number of Fits Required": [10, 5, 5, len(X_small)],
})
strategy_results["Mean Accuracy"] = strategy_results["Mean Accuracy"].round(4)
strategy_results["Std Dev Across Folds/Splits"] = strategy_results["Std Dev Across Folds/Splits"].round(5)
strategy_results